# Sparse Walker: temporal skip memories on ML-1M

Diagnostic experiment for the long-history plateau. The base state remains **K=8**, degree=4, 2 graph hops. We add only three frozen-in-time sparse landmarks: short, medium and long.

The experiment warm-starts from the best plain ML-1M Walker checkpoint, disables new pursuit rewires by default, and uses the same canonical full-catalog evaluator/fingerprint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!rm -rf /content/Sparsewalker
!git clone -q -b agent/walker-temporal-skips https://github.com/hanialshater/Sparsewalker-.git /content/Sparsewalker
%cd /content/Sparsewalker
!pip -q install -e .
import torch
print('torch', torch.__version__)
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('bf16', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 1. Fast shape/backward smoke test

This catches wiring/autograd errors before using the real checkpoint.

In [ ]:
from sparsewalker.models import SparseWalkerTemporalMemory
m=SparseWalkerTemporalMemory(3706,200,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4).cuda()
x=torch.randint(1,3707,(4,40),device='cuda')
with torch.autocast('cuda',dtype=torch.bfloat16):
    H=m(x)
    loss=H.float().square().mean()
loss.backward()
print('SMOKE OK', H.shape, 'loss', float(loss))
del m,x,H,loss
torch.cuda.empty_cache()

## 2. Run the temporal-skip diagnostic

Expected base checkpoint: `/content/drive/MyDrive/sparsewalker_canonical_pair/ml1m/seed42/SparseWalker_FullCE/best.pt`. Results go to a separate directory, so the canonical Walker run is untouched.

In [ ]:
!python experiments/run_ml1m_temporal_skips.py \
  --seed 42 \
  --max-epochs 20 \
  --eval-every 2 \
  --patience 8 \
  --batch-size 128 \
  --eval-batch-size 1024

## 3. Inspect the learning curve

In [ ]:
import pandas as pd
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_temporal_skips/ml1m/seed42/history.csv')
if p.exists():
    df=pd.read_csv(p)
    display(df[['epoch','loss','NDCG@10','HR@10','MRR@10','memory_share','seconds','positions_per_s']])
    print('best temporal val NDCG@10', df['NDCG@10'].max())
else:
    print('No history yet')